In [1]:
# Import config
from pathlib import Path
import sys

p = Path.cwd()
for _ in range(6):
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break
    p = p.parent

from config import *

# create main dirs + language subfolders
ensure_dirs(
    DATA_DIR, RAW_DIR, RESULT_DIR, MODELS_DIR, LOG_DIR, CHARACTER_DIR, TEST_DATA_DIR
)
ensure_dirs(*(RAW_DIR / lang for lang in LANGUAGES))
ensure_dirs(*(RESULT_DIR / lang for lang in LANGUAGES))

# Define data/result directory

In [2]:
import os

data_folder = TEST_DATA_DIR
result_folder = RESULT_DIR
manga_list = "vi/Almark/Vol. 1 Ch. 1"  # Will be changed to list later

# Almark
manga_folder = os.path.join(data_folder, manga_list)

individual_result_folder = os.path.join(result_folder, manga_list)
json_output_dir = os.path.join(individual_result_folder, "json_results")
result_image_output_dir = os.path.join(individual_result_folder, MAGI_IMAGE_RESULT)

cut_bubbles_dir = os.path.join(individual_result_folder, CUT_BUBBLES)

raw_images = os.listdir(manga_folder)
json_files = os.listdir(json_output_dir)
google_lens_result_dir = os.path.join(individual_result_folder, "google_lens_result")
os.makedirs(google_lens_result_dir, exist_ok=True)

In [3]:
import os
import json
from chrome_lens_py import LensAPI


# -------------------------------
# NEW: Google Lens OCR function
# -------------------------------
async def get_transcript_from_image(img_path, api):
    """
    Run OCR using Google Lens (Chrome Lens API)
    """
    try:
        result = await api.process_image(image_path=img_path, ocr_language="vi")
        text = result.get("ocr_text", "")
        return " ".join(text.split()) if text.strip() else ""
    except Exception as e:
        print(f"⚠️ Lens OCR failed on {img_path}: {e}")
        return ""


# -------------------------------
# UPDATED: process_json_file
# -------------------------------
async def process_json_file(
    json_file,
    json_output_dir,
    cut_bubbles_dir,
    google_lens_result_dir,
    api,
):
    base_name = os.path.splitext(json_file)[0]
    json_path = os.path.join(json_output_dir, json_file)
    cut_page_dir = os.path.join(cut_bubbles_dir, base_name)

    if not os.path.exists(cut_page_dir):
        print(f"⚠️ No cut_bubbless found for {json_file}, skipping...")
        return

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    new_ocr = []
    for idx, (_bbox, is_essential) in enumerate(
        zip(data["texts"], data["is_essential_text"])
    ):
        # if not is_essential:
        #     new_ocr.append(data["ocr"][idx])
        #     continue

        cut_img_path = os.path.join(cut_page_dir, f"{base_name}_{idx:03}.png")
        if not os.path.exists(cut_img_path):
            print(f"⚠️ Missing cut image {cut_img_path}, keeping original OCR.")
            new_ocr.append(data["ocr"][idx])
            continue

        try:
            vi_text = await get_transcript_from_image(cut_img_path, api)
            if not vi_text:
                vi_text = data["ocr"][idx]
        except Exception as e:
            print(f"❌ OCR pipeline failed for {cut_img_path}: {e}")
            vi_text = data["ocr"][idx]

        new_ocr.append(vi_text)

    data["ocr"] = new_ocr

    out_path = os.path.join(google_lens_result_dir, json_file)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"✅ Saved result for {json_file} → {out_path}")


# -------------------------------
# MAIN LOOP (async wrapper)
# -------------------------------
async def main():
    api = LensAPI()

    for json_file in json_files:
        if json_file.endswith(".json"):
            await process_json_file(
                json_file,
                json_output_dir,
                cut_bubbles_dir,
                google_lens_result_dir,
                api,
            )


await main()

✅ Saved result for 00.json → D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\google_lens_result\00.json
✅ Saved result for 01.json → D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\google_lens_result\01.json
✅ Saved result for 02.json → D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\google_lens_result\02.json
✅ Saved result for 03.json → D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\google_lens_result\03.json
✅ Saved result for 04.json → D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\google_lens_result\04.json
✅ Saved result for 05.json → D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\google_lens_result\05.json
✅ Saved result for 06.json → D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\google_lens_result\

Individual test

In [ ]:
# Test multiple lines
import os

# test_image_dir = "../data/vi/test_data/multiple_line"
test_image_dir = "../results/vi/Almark/Vol. 1 Ch. 1/cut_bubbless/47"
test_images = os.listdir(test_image_dir)

for test_image in test_images:
    img_path = os.path.join(test_image_dir, test_image)

    # 🔄 Reuse the function
    final_text = get_transcript_from_image(img_path, reader, detector)

    print(f"{test_image}: {final_text if final_text else '(empty)'}")

47_001.png: VỚI TÌNH THẾ ĐÓ, RAIZ KHÔNG THỂ NÀO RỜI KHỎI ĐỘI
47_002.png: VĨ VẬY THAY VĨ DỰ LỄ NHẬP HỌC, ALMARK ĐÃ PHẢI RA CHIẾN TRƯỜNG...
47_003.png: ALMARK LẦN ĐẦU TIÊN RA CHIẾN TRẬN
47_004.png: TRẬN SAU TIẾP TỤC RA TRẬN


In [ ]:
# Test one line
import os

test_image_dir = "../data/vi/test_data/one_line"
test_images = os.listdir(test_image_dir)

for test_image in test_images:
    img_path = os.path.join(test_image_dir, test_image)

    # 🔄 Reuse the function
    final_text = get_transcript_from_image(img_path, reader, detector)

    print(f"{test_image}: {final_text if final_text else '(empty)'}")

04_001.png: VÌ CON LÀ
05_004.png: BÌNH TĨNH
